# 09 学习率：每一步参数走多远？

## 知识点标注
- 行业状态：基础标准和常见工程实践。
- 现实用途：控制每次参数更新的步长。
- 本课类型：本地数学演示，不使用 GPU，不训练 MokioMind。

前面已经知道梯度告诉参数往哪边走；学习率决定这一步走多远。

## 先看最简单的更新

```text
新参数 = 旧参数 - 学习率 × 梯度

梯度 = -4，学习率 = 0.1
更新量 = -0.1 × (-4) = +0.4
w: 1.0 -> 1.4
```

同一个梯度：学习率太大，可能跨过最低点；学习率太小，训练会很慢。

In [1]:
import math

def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))

base_lr = 1.0
total_steps = 10
for step in [0, 1, 2, 5, 8, 10]:
    print(f'step={step:2d}, lr={get_lr(step, total_steps, base_lr):.4f}')

step= 0, lr=1.0000
step= 1, lr=0.9780
step= 2, lr=0.9141
step= 5, lr=0.5500
step= 8, lr=0.1859
step=10, lr=0.1000


### 先预测

1. 第 0 步的学习率大约是多少？
2. 第 10 步的学习率大约是多少？
3. 这段函数是在改变梯度方向，还是改变每次更新的步长？

## 这条曲线在做什么？

项目使用余弦函数让学习率从接近初始值逐渐下降到初始值的 10%。这是人为选择的训练策略，不是模型自动学出来的知识。

```text
学习率
  ↑  初始值 ─────╮
  │             ╲
  │              ╲  逐渐变小
  │               ╲
  │  10% 初始值 ───╰────→ 训练步数
```

## 对应 MokioMind 真实代码

训练循环每一步先计算学习率：

```python
lr = get_lr(epoch * iters + step, args.epochs * iters, args.learning_rate)

for param_group in optimizer.param_groups:
    param_group["lr"] = lr
```

第一行得到当前训练进度对应的学习率；第二段把它写进优化器管理的参数组。后面的 `scaler.step(optimizer)` 才使用这个学习率更新参数。

真实位置：[trainer/train_pretrain.py:48](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:48) 和 [trainer/train_pretrain.py:50](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:50)。公式实现位于 [trainer/trainer_utils.py:22](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/trainer_utils.py:22)。

## 验收题

1. 学习率控制的是梯度方向，还是参数每次移动的步长？
2. 如果梯度为 `-4`、学习率从 `0.1` 变成 `0.01`，一次更新量会怎样变化？
3. 项目中的 `get_lr(...)` 是模型学出来的，还是人为设计的训练策略？
4. 真实代码中哪一行把学习率写入优化器？